# Skin Lesion Bias Reduction — cGAN training (Colab)

Trains the conditional DCGAN-style GAN at `src/train.py` on Fitzpatrick17k using a Colab GPU.

**Order of operations**
1. Confirm GPU + mount Drive
2. Point the notebook at your code + data
3. Cache images locally for fast IO
4. Pull the latest repo from `origin/main`
5. Install dependencies
6. Configure cGAN hyperparameters
7. Train the cGAN
8. *(optional)* Resume from a checkpoint
9. Watch progress in TensorBoard
10. Generate synthetic samples for underrepresented (FST 5/6) cells

## 1. Verify GPU and Colab environment

In [ ]:
import sys

IN_COLAB = "google.colab" in sys.modules
print("Colab:", IN_COLAB)

import torch
print("torch:", torch.__version__, "cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    !nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv

## 2. Connect your code and data

Edit `PROJECT_ROOT`, `DATASET_CSV`, and `IMAGE_DIR` to match where the repo and dataset live on your Drive. The paths below assume the same layout as `colab_training.ipynb`.

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

In [ ]:
from pathlib import Path

# === EDIT THESE ===
PROJECT_ROOT = Path("/content/drive/MyDrive/SkinLesionBiasReduction")
IMAGE_DIR    = PROJECT_ROOT / "dataset/images"
DATASET_CSV  = PROJECT_ROOT / "dataset/fitzpatrick17k_cleaned.csv"
# ==================

print("PROJECT_ROOT:", PROJECT_ROOT, "exists:", PROJECT_ROOT.exists())
print("IMAGE_DIR:   ", IMAGE_DIR,    "exists:", IMAGE_DIR.exists())
print("DATASET_CSV: ", DATASET_CSV,  "exists:", DATASET_CSV.exists())
assert PROJECT_ROOT.exists(), f"PROJECT_ROOT does not exist: {PROJECT_ROOT}"
assert IMAGE_DIR.exists(),    f"IMAGE_DIR does not exist:    {IMAGE_DIR}"
assert DATASET_CSV.exists(),  f"DATASET_CSV does not exist:  {DATASET_CSV}"

%cd $PROJECT_ROOT

### 2.1 Cache the images on local disk

Reading 16k JPEGs through Drive's FUSE mount is the throughput bottleneck on Colab. Copy them once to `/content/local_images` (parallel `cp -n`, resumable) and point the trainer at the local copy. The dataset CSV stays on Drive — it's a single small file.

In [ ]:
import subprocess, time

DRIVE_IMAGE_DIR = IMAGE_DIR
LOCAL_IMAGE_DIR = Path("/content/local_images")
LOCAL_IMAGE_DIR.mkdir(parents=True, exist_ok=True)

def _count_files(d):
    r = subprocess.run(f"ls -1 '{d}' 2>/dev/null | wc -l",
                       shell=True, capture_output=True, text=True)
    return int(r.stdout.strip() or 0)

n_source = _count_files(DRIVE_IMAGE_DIR)
n_local  = _count_files(LOCAL_IMAGE_DIR)
print(f"Drive: {n_source} files | Local: {n_local} files "
      f"(need ~{max(0, n_source - n_local)} more)")

if n_local >= n_source > 0:
    print("Local cache is complete — skipping copy.")
else:
    print("Parallel-copying from Drive (64 workers)...")
    t0 = time.time()
    cmd = (
        f"cd '{DRIVE_IMAGE_DIR}' && "
        f"find . -maxdepth 1 -type f -print0 | "
        f"xargs -0 -n 64 -P 64 cp -n -t '{LOCAL_IMAGE_DIR}/'"
    )
    subprocess.run(cmd, shell=True, check=True)
    n_local = _count_files(LOCAL_IMAGE_DIR)
    print(f"Done in {time.time()-t0:.1f}s — local now has {n_local} files.")

# Use the local cache for training
TRAIN_IMAGE_DIR = LOCAL_IMAGE_DIR

### 2.2 Pull the latest repo state

Discards local edits in the Drive checkout and resets to `origin/main`. Skip this cell if you have uncommitted work on Drive you don't want overwritten.

In [ ]:
!git fetch origin main && git reset --hard origin/main

## 3. Install dependencies

Colab images already include `torch`, `torchvision`, `numpy`, `pandas`, `Pillow`, `tqdm`, `matplotlib`, `scipy`, and `tensorboard`. The cGAN trainer uses only those — no extra installs needed. The cell below is a no-op safety net in case Colab ever ships a slimmer image.

In [ ]:
!pip install --quiet 'tensorboard' 'scipy'

## 4. Configure cGAN hyperparameters

Defaults match `src/train.py`. Adjust `EPOCHS` and `BATCH_SIZE` to your GPU:
- T4 (~15 GB): `BATCH_SIZE = 64` is comfortable
- A100 (~40 GB): `BATCH_SIZE = 128`+ if you want faster epochs

Outputs land under `PROJECT_ROOT/outputs/<timestamp>/{checkpoints,samples,logs}`. Keeping outputs on Drive means checkpoints survive a runtime disconnect.

In [ ]:
EPOCHS              = 200
BATCH_SIZE          = 64
LR                  = 2e-4
BETA1               = 0.5
BETA2               = 0.999
LATENT_DIM          = 100
EMBEDDING_DIM       = 50
NGF                 = 64
NDF                 = 64
LABEL_SMOOTHING     = 0.1
NOISE_STD           = 0.1
NOISE_DECAY         = 0.995
CHECKPOINT_INTERVAL = 10
SAMPLE_INTERVAL     = 5
FID_INTERVAL        = 20
FID_NUM_SAMPLES     = 1000
NUM_WORKERS         = 4

OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Outputs will land under:", OUTPUT_DIR)

## 5. Train the cGAN

Logs go to stdout *and* TensorBoard. A new `outputs/<YYYYMMDD_HHMMSS>/` directory is created at start; record its name from the first few lines of output if you want to resume later.

> **Smoke test:** set `EPOCHS = 1` (and optionally `FID_INTERVAL = 0`) to verify the pipeline before committing to a 200-epoch run.

In [ ]:
!python src/train.py \
    --csv_path "{DATASET_CSV}" \
    --image_dir "{TRAIN_IMAGE_DIR}" \
    --epochs {EPOCHS} \
    --batch_size {BATCH_SIZE} \
    --lr {LR} \
    --beta1 {BETA1} \
    --beta2 {BETA2} \
    --latent_dim {LATENT_DIM} \
    --embedding_dim {EMBEDDING_DIM} \
    --ngf {NGF} \
    --ndf {NDF} \
    --label_smoothing {LABEL_SMOOTHING} \
    --noise_std {NOISE_STD} \
    --noise_decay {NOISE_DECAY} \
    --checkpoint_interval {CHECKPOINT_INTERVAL} \
    --sample_interval {SAMPLE_INTERVAL} \
    --fid_interval {FID_INTERVAL} \
    --fid_num_samples {FID_NUM_SAMPLES} \
    --num_workers {NUM_WORKERS} \
    --output_dir "{OUTPUT_DIR}" \
    --device cuda

## 6. (Optional) Resume from a checkpoint

`src/train.py --resume <path>` re-hydrates both networks and both optimizers and continues from the saved epoch. Point `RESUME_CKPT` at a `checkpoint_epoch_XXXX.pt` (or `final_model.pt`) under `outputs/<run>/checkpoints/`.

In [ ]:
# RESUME_CKPT = OUTPUT_DIR / "20260501_120000/checkpoints/checkpoint_epoch_0040.pt"
# !python src/train.py \
#     --csv_path "{DATASET_CSV}" \
#     --image_dir "{TRAIN_IMAGE_DIR}" \
#     --resume "{RESUME_CKPT}" \
#     --epochs {EPOCHS} \
#     --batch_size {BATCH_SIZE} \
#     --num_workers {NUM_WORKERS} \
#     --output_dir "{OUTPUT_DIR}" \
#     --device cuda

## 7. TensorBoard

Loss curves, D(x) / D(G(z)) traces, sample grids per `SAMPLE_INTERVAL`, and FID per `FID_INTERVAL`.

In [ ]:
%load_ext tensorboard

In [ ]:
%tensorboard --logdir $OUTPUT_DIR

## 8. Preview the latest sample grid

Pulls the most recent `samples_epoch_XXXX.png` from the freshest run.

In [ ]:
from IPython.display import Image as IPyImage, display

runs = sorted(OUTPUT_DIR.glob("*/samples"), key=lambda p: p.stat().st_mtime, reverse=True)
if not runs:
    print("No samples yet.")
else:
    grids = sorted(runs[0].glob("samples_epoch_*.png"))
    if not grids:
        print(f"No grids in {runs[0]} yet.")
    else:
        latest = grids[-1]
        print("Latest grid:", latest)
        display(IPyImage(filename=str(latest)))

## 9. Generate synthetic images for underrepresented skin tones

After training, point `GEN_CKPT` at the checkpoint you want to sample from and run `src/generate.py`. Defaults below produce 500 images each for FST 5 and FST 6 across all three lesion types, plus a 4×4 preview grid and a metadata CSV that follows the Fitzpatrick17k schema (so it can be merged into a combined dataloader for the downstream classifier).

In [ ]:
ckpts = sorted(OUTPUT_DIR.glob("*/checkpoints/*.pt"), key=lambda p: p.stat().st_mtime, reverse=True)
assert ckpts, f"No checkpoints under {OUTPUT_DIR}"
GEN_CKPT = ckpts[0]
print("Generating from:", GEN_CKPT)

GEN_OUTPUT_DIR  = PROJECT_ROOT / "generated_images"
GEN_NUM_SAMPLES = 500
GEN_TARGET_FST  = "5 6"  # Fitzpatrick scale (1-6); 5 & 6 are the underrepresented cells

In [ ]:
!python src/generate.py \
    --checkpoint "{GEN_CKPT}" \
    --output_dir "{GEN_OUTPUT_DIR}" \
    --num_samples {GEN_NUM_SAMPLES} \
    --target_skin_tones {GEN_TARGET_FST} \
    --target_lesion_types benign malignant non-neoplastic \
    --batch_size 64 \
    --save_grid \
    --create_csv \
    --device cuda